In [ ]:

# Install Optuna
!pip install -q optuna

# Import Optuna
import optuna

# Verify installation
print("Optuna version:", optuna.__version__)
print("Optuna imported successfully!")


Optuna version: 5.0.0
Optuna imported successfully!


In [ ]:
# Install + Upload dataset
!pip install -q albumentations optuna einops timm fvcore huggingface_hub

import os, glob, zipfile
from google.colab import files

ROOT = "/content/data/IDRiD"
os.makedirs(ROOT, exist_ok=True)

print("Select your dataset zip (B__Disease_Grading.zip) ...")
uploaded = files.upload()
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, "r") as z:
        z.extractall(ROOT)
    os.remove(fname)
    print("Extracted:", fname)


csvs = glob.glob(f"{ROOT}/**/*.csv", recursive=True)
imgs = glob.glob(f"{ROOT}/**/*.jpg", recursive=True)
print("CSV files:", *csvs, sep="\n  ")
print("Images:", len(imgs))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Select your dataset zip (B__Disease_Grading.zip) ...


Saving B. Disease Grading.zip to B. Disease Grading.zip
Extracted: B. Disease Grading.zip
CSV files:
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Images: 516


In [ ]:
#  Locate dataset folders
import os, glob

ROOT = "/content/data/IDRiD"
csv_train = glob.glob(f"{ROOT}/**/*Training Labels*.csv", recursive=True)[0]
csv_test  = glob.glob(f"{ROOT}/**/*Testing Labels*.csv",  recursive=True)[0]
img_train_dir = [d for d in glob.glob(f"{ROOT}/**/a. Training Set", recursive=True) if "Original Images" in d][0]
img_test_dir  = [d for d in glob.glob(f"{ROOT}/**/b. Testing Set",  recursive=True) if "Original Images" in d][0]

print("Train CSV :", csv_train)
print("Test CSV  :", csv_test)
print("Train imgs:", img_train_dir, len(os.listdir(img_train_dir)))   # 413
print("Test imgs :", img_test_dir,  len(os.listdir(img_test_dir)))    # 103

Train CSV : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Test CSV  : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
Train imgs: /content/data/IDRiD/B. Disease Grading/1. Original Images/a. Training Set 413
Test imgs : /content/data/IDRiD/B. Disease Grading/1. Original Images/b. Testing Set 103


In [ ]:
# Configuration + seed
import random, numpy as np, torch

class CFG:
    img_size      = 224
    val_fraction  = 0.20
    split_seed    = 42
    seed          = 42
    retina_thr    = 10
    feat_dim      = 512
    n_dr, n_dme   = 5, 3
    lr            = 1e-4
    batch_size    = 32
    weight_decay  = 1e-4
    dropout       = 0.30
    epochs        = 100
    lambda_dr     = 1.0
    lambda_dme    = 1.0
    ln_eps        = 1e-6
    drop_path     = 0.2
    num_workers   = 2

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cpu 


In [ ]:
#  Labels + data splitting
import pandas as pd, hashlib
from sklearn.model_selection import train_test_split

def read_labels(csv_path, img_dir, part):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df[["Image name", "Retinopathy grade", "Risk of macular edema"]].dropna()   # drops empty extra columns
    df.columns = ["name", "dr", "dme"]
    df["name"] = df["name"].astype(str).str.strip()
    df[["dr", "dme"]] = df[["dr", "dme"]].astype(int)
    df["path"] = [glob.glob(os.path.join(img_dir, n + ".*"))[0] for n in df["name"]]
    df["part"] = part
    return df.reset_index(drop=True)

dev_df  = read_labels(csv_train, img_train_dir, "train")   # 413
test_df = read_labels(csv_test,  img_test_dir,  "test")    # 103

tr_idx, va_idx = train_test_split(np.arange(len(dev_df)), test_size=CFG.val_fraction,
                                  random_state=CFG.split_seed, shuffle=True)
train_df = dev_df.iloc[tr_idx].reset_index(drop=True)
val_df   = dev_df.iloc[va_idx].reset_index(drop=True)

def class_table(train, val, test):
    rows = {}
    for n, d in (("Training", train), ("Validation", val), ("Held-out Test", test)):
        r = {"Images": len(d)}
        r.update({f"DR {g}": int((d.dr == g).sum()) for g in range(5)})
        r.update({f"DME {g}": int((d.dme == g).sum()) for g in range(3)})
        rows[n] = r
    t = pd.DataFrame(rows).T; t.loc["Total"] = t.sum(); return t

table1 = class_table(train_df, val_df, test_df)
display(table1)

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
dev_hashes = {md5(p) for p in dev_df.path}
print("Duplicate images between development and test:", sum(md5(p) in dev_hashes for p in test_df.path))

os.makedirs("/content/outputs", exist_ok=True)
for n, d in (("train", train_df), ("val", val_df), ("test", test_df)):
    d[["name", "dr", "dme"]].to_csv(f"/content/outputs/split_{n}.csv", index=False)
table1.to_csv("/content/outputs/table1_class_distribution.csv")

,Images,DR 0,DR 1,DR 2,DR 3,DR 4,DME 0,DME 1,DME 2
Training,330,111,13,108,58,40,144,32,154
Validation,83,23,7,28,16,9,33,9,41
Held-out Test,103,34,5,32,19,13,45,10,48
Total,516,168,25,168,93,62,222,51,243


Duplicate images between development and test: 1


In [ ]:
# Preprocessing
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def retina_bbox(rgb, thr=CFG.retina_thr):
    mask = (rgb.max(axis=2) > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        h, w = mask.shape
        return 0, 0, w, h
    return cv2.boundingRect(max(cnts, key=cv2.contourArea))


def load_and_preprocess(path, size=CFG.img_size):
    """Retina crop -> resize 224 x 224 (used by the Dataset class)."""
    rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    x, y, w, h = retina_bbox(rgb)
    return cv2.resize(rgb[y:y + h, x:x + w], (size, size), interpolation=cv2.INTER_AREA)


def preprocess_steps(path, size=CFG.img_size):
    """Return (original, cropped, resized) for one image."""
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(f"Could not read: {path}")
    raw = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x, y, w, h = retina_bbox(raw)
    crop = raw[y:y + h, x:x + w]
    resized = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)
    return raw, crop, resized


In [ ]:
#  Augmentation + Dataset + Loaders

import albumentations as A
import torch
import numpy as np
import random
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Rotate(
        limit=10,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5
    ),

    A.Affine(
        scale=(0.9, 1.1),
        translate_percent={
            "x": (-0.05, 0.05),
            "y": (-0.05, 0.05)
        },
        shear=(-3, 3),
        p=0.4
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.4
    ),
])


class IDRiDGrading(Dataset):

    def __init__(self, df, aug=None):
        self.df = df.reset_index(drop=True)
        self.aug = aug

        self.cache = [
            load_and_preprocess(p)
            for p in self.df.path
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):

        img = self.cache[i]

        if self.aug is not None:
            img = self.aug(image=img)["image"]

        x = torch.from_numpy(
            (img.astype(np.float32) / 255.0)
            .transpose(2, 0, 1)
            .copy()
        )

        return (
            x,
            int(self.df.dr[i]),
            int(self.df.dme[i]),
            self.df.name[i]
        )


def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


train_ds = IDRiDGrading(train_df, aug=train_aug)
val_ds   = IDRiDGrading(val_df)
test_ds  = IDRiDGrading(test_df)


g = torch.Generator()
g.manual_seed(CFG.seed)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)


print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Testing samples:", len(test_ds))




Training samples: 330
Validation samples: 83
Testing samples: 103


In [ ]:

import os, glob, math, json, time, copy, random, warnings
from collections import OrderedDict
from types import SimpleNamespace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint as cp
from sklearn.metrics import f1_score, cohen_kappa_score, matthews_corrcoef
from scipy.stats import binom, mannwhitneyu
import timm
warnings.filterwarnings("ignore")


assert "load_and_preprocess" in globals(), "Add load_and_preprocess() to your Preprocessing cell (see note)."
assert all(n in globals() for n in ("train_ds", "val_ds", "test_ds", "CFG", "device", "set_seed", "seed_worker"))


from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/DR_DME_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

USE_AMP = device.type == "cuda"
FINAL_HP = dict(lr=1e-4, batch_size=32, weight_decay=1e-4, dropout=0.30, epochs=100)   # AHEO-selected configuration

# ImageNet EfficientVMamba-S weights
BACKBONE_CKPT = None
def resolve_backbone_ckpt():
    global BACKBONE_CKPT
    ck = glob.glob("/content/weights/evm_small/**/*.ckpt", recursive=True)
    if not ck:
        from huggingface_hub import snapshot_download
        snapshot_download("TerryPei/EfficientVMamba-Small", local_dir="/content/weights/evm_small")
        ck = glob.glob("/content/weights/evm_small/**/*.ckpt", recursive=True)
    BACKBONE_CKPT = ck[0] if ck else None
    print("Backbone checkpoint:", BACKBONE_CKPT)
    return BACKBONE_CKPT

# reuse the datasets already built by your notebook cells
D = SimpleNamespace(train_df=train_df, val_df=val_df, test_df=test_df,
                    train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)
def build_data(): return D
print("Setup OK | train/val/test =", len(D.train_ds), len(D.val_ds), len(D.test_ds))


Mounted at /content/drive
Setup OK | train/val/test = 330 83 103


In [ ]:
# EfficientVMamba-S BACKBONE
USE_CHECKPOINT = True

def selective_scan(u, delta, A_, B, C, D, delta_bias, softplus=True):
    with torch.autocast(device_type="cuda", enabled=False):
        u, delta, A_, B, C, D, delta_bias = [t.float() for t in (u, delta, A_, B, C, D, delta_bias)]
        b, KD, L = u.shape; K, N = B.shape[1], A_.shape[1]; d = KD // K
        delta = delta + delta_bias[None, :, None]
        if softplus: delta = F.softplus(delta)
        Bt = B.permute(0, 1, 3, 2).unsqueeze(2); Ct = C.permute(0, 1, 3, 2)
        dA  = torch.exp(delta.view(b, K, d, L, 1) * A_.view(1, K, d, 1, N))
        dBu = (delta * u).view(b, K, d, L, 1) * Bt
        h = u.new_zeros(b, K, d, N); hs = []
        for t in range(L):
            h = torch.addcmul(dBu[:, :, :, t], dA[:, :, :, t], h); hs.append(h)
        hs = torch.stack(hs, 3)
        y = torch.einsum("bkdln,bkln->bkdl", hs, Ct).reshape(b, KD, L)
        return y + u * D[None, :, None]

def efficient_scan(x, s=2):
    B, C, H, W = x.shape
    if H % s or W % s: x = F.pad(x, (0, (-W) % s, 0, (-H) % s))
    xt = x.transpose(2, 3)
    return torch.stack([x[:, :, ::s, ::s].reshape(B, C, -1), xt[:, :, ::s, 1::s].reshape(B, C, -1),
                        x[:, :, ::s, 1::s].reshape(B, C, -1), xt[:, :, 1::s, 1::s].reshape(B, C, -1)], 1)

def efficient_merge(ys, H0, W0, s=2):
    B, K, C, L = ys.shape; H, W = math.ceil(H0 / s), math.ceil(W0 / s)
    y = ys.new_zeros(B, C, H * s, W * s)
    y[:, :, ::s, ::s]   = ys[:, 0].reshape(B, C, H, W)
    y[:, :, 1::s, ::s]  = ys[:, 1].reshape(B, C, W, H).transpose(2, 3)
    y[:, :, ::s, 1::s]  = ys[:, 2].reshape(B, C, H, W)
    y[:, :, 1::s, 1::s] = ys[:, 3].reshape(B, C, W, H).transpose(2, 3)
    return y[:, :, :H0, :W0].reshape(B, C, -1)

class Permute(nn.Module):
    def __init__(self, *d): super().__init__(); self.d = d
    def forward(self, x): return x.permute(*self.d)

class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        m = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(1 - self.p)
        return x * m / (1 - self.p)

class SS2D(nn.Module):
    def __init__(self, d_model, d_state=16, ssm_ratio=2.0, step=2):
        super().__init__()
        di = int(ssm_ratio * d_model); R = math.ceil(d_model / 16); N, K = d_state, 4
        self.R, self.N, self.K, self.step = R, N, K, step
        self.in_proj = nn.Linear(d_model, 2 * di, bias=False); self.act = nn.SiLU()
        self.conv2d = nn.Conv2d(di, di, 3, padding=1, groups=di)
        xw = torch.empty(K, R + 2 * N, di)
        for k in range(K): nn.init.kaiming_uniform_(xw[k], a=math.sqrt(5))
        self.x_proj_weight = nn.Parameter(xw)
        self.dt_projs_weight = nn.Parameter(torch.empty(K, di, R).uniform_(-R ** -0.5, R ** -0.5))
        dt = torch.exp(torch.rand(K, di) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
        self.dt_projs_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        self.A_logs = nn.Parameter(torch.log(torch.arange(1, N + 1, dtype=torch.float32).repeat(K * di, 1)))
        self.Ds = nn.Parameter(torch.ones(K * di))
        self.out_norm = nn.LayerNorm(di); self.out_proj = nn.Linear(di, d_model, bias=False)
    def forward(self, x):
        B, H, W, _ = x.shape
        x, z = self.in_proj(x).chunk(2, -1); z = self.act(z)
        x = self.act(self.conv2d(x.permute(0, 3, 1, 2).contiguous())); D = x.shape[1]
        xs = efficient_scan(x, self.step); L = xs.shape[-1]
        dts, Bs, Cs = torch.split(torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight), [self.R, self.N, self.N], 2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_projs_weight)
        ys = selective_scan(xs.reshape(B, -1, L), dts.reshape(B, -1, L), -torch.exp(self.A_logs.float()),
                            Bs, Cs, self.Ds, self.dt_projs_bias.reshape(-1), True).reshape(B, 4, D, L)
        y = self.out_norm(efficient_merge(ys, H, W, self.step).transpose(1, 2)).reshape(B, H, W, D).to(z.dtype)
        return self.out_proj(y * z)

class BiAttn(nn.Module):
    def __init__(self, c, r=0.125):
        super().__init__()
        self.norm = nn.LayerNorm(c); self.global_reduce = nn.Linear(c, int(c * r))
        self.act_fn = nn.GELU(); self.channel_select = nn.Linear(int(c * r), c)
    def forward(self, x):
        g = self.norm(x).mean([1, 2], keepdim=True)
        return x * torch.sigmoid(self.channel_select(self.act_fn(self.global_reduce(g))))

class EVSSBlock(nn.Module):
    def __init__(self, dim, dp=0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim); self.op = SS2D(dim)
        self.conv_branch = nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1, groups=dim), nn.BatchNorm2d(dim),
                                         nn.GELU(), nn.Conv2d(dim, dim, 1))
        self.se = BiAttn(dim); self.drop_path = DropPath(dp)
    def _forward(self, inp):
        x = self.norm(inp)
        xc = self.conv_branch(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        return inp + self.drop_path(self.se(self.op(x)) + self.se(xc))
    def forward(self, inp):
        if USE_CHECKPOINT and self.training and torch.is_grad_enabled():
            return cp.checkpoint(self._forward, inp, use_reentrant=False)
        return self._forward(inp)

class SqueezeExcite(nn.Module):
    def __init__(self, c, r):
        super().__init__()
        self.avgp = nn.AdaptiveAvgPool2d(1); self.conv_reduce = nn.Conv2d(c, r, 1)
        self.act_fn = nn.ReLU(inplace=True); self.conv_expand = nn.Conv2d(r, c, 1)
    def forward(self, x): return x * torch.sigmoid(self.conv_expand(self.act_fn(self.conv_reduce(self.avgp(x)))))

class InvertedResidual(nn.Module):
    def __init__(self, c, e=4, se=0.125, dp=0.):
        super().__init__(); m = c * e
        self.conv_pw  = nn.Sequential(nn.Conv2d(c, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.conv_dw  = nn.Sequential(nn.Conv2d(m, m, 3, 1, 1, groups=m, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.se       = SqueezeExcite(m, int(m * se))
        self.conv_pw2 = nn.Sequential(nn.Conv2d(m, c, 1, bias=False), nn.BatchNorm2d(c))
        self.drop = DropPath(dp)
    def forward(self, x): return x + self.drop(self.conv_pw2(self.se(self.conv_dw(self.conv_pw(x)))))

class PatchMerging2D(nn.Module):
    def __init__(self, d, od):
        super().__init__(); self.reduction = nn.Linear(4 * d, od, bias=False); self.norm = nn.LayerNorm(4 * d)
    def forward(self, x):
        x = torch.cat([x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]], -1)
        return self.reduction(self.norm(x))

class EfficientVMambaS(nn.Module):
    def __init__(self, depths=(2, 2, 4, 2), dims=(64, 128, 256, 512), drop_path=0.2):
        super().__init__()
        self.patch_embed = nn.Sequential(nn.Conv2d(3, dims[0], 4, 4), Permute(0, 2, 3, 1), nn.LayerNorm(dims[0]))
        dpr = torch.linspace(0, drop_path, sum(depths)).tolist(); self.layers = nn.ModuleList()
        for i, dep in enumerate(depths):
            dp = dpr[sum(depths[:i]):sum(depths[:i + 1])]
            down = PatchMerging2D(dims[i], dims[i + 1]) if i < 3 else nn.Identity()
            if i < 2:
                self.layers.append(nn.Sequential(OrderedDict(
                    blocks=nn.Sequential(*[EVSSBlock(dims[i], dp[j]) for j in range(dep)]), downsample=down)))
            else:
                self.layers.append(nn.Sequential(Permute(0, 3, 1, 2),
                                                 *[InvertedResidual(dims[i], dp=dp[j]) for j in range(dep)],
                                                 Permute(0, 2, 3, 1), down))
        self.classifier = nn.Sequential(OrderedDict(norm=nn.LayerNorm(dims[-1])))
    def forward(self, x):
        x = self.patch_embed(x)
        for l in self.layers: x = l(x)
        return self.classifier.norm(x).permute(0, 3, 1, 2).contiguous()      # B,C,7,7

_CKPT_CACHE = {}
def load_imagenet_weights(model, path):
    if not path or not os.path.exists(path):
        if "warned" not in _CKPT_CACHE: print("WARNING: no ImageNet weights -> random init"); _CKPT_CACHE["warned"] = 1
        return model
    if path not in _CKPT_CACHE:                                                   # read the checkpoint once
        ck = torch.load(path, map_location="cpu", weights_only=False)
        sd = ck.get("model", ck.get("state_dict", ck))
        _CKPT_CACHE[path] = {k.replace("module.", ""): v for k, v in sd.items()}
    own = model.state_dict()
    sd = {k: v for k, v in _CKPT_CACHE[path].items() if k in own and own[k].shape == v.shape}
    model.load_state_dict(sd, strict=False)
    return model



In [ ]:
#  MULTI-TASK MODEL
class TokenAttention(nn.Module):
    """Single-head scaled dot-product attention, pre-LayerNorm (Eqs. 10-11)."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.ln_q, self.ln_kv = nn.LayerNorm(d, eps=eps), nn.LayerNorm(d, eps=eps)
        self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.scale = 1 / math.sqrt(d)
    def forward(self, q_src, kv_src):
        q = self.q(self.ln_q(q_src)); kv = self.ln_kv(kv_src)
        w = torch.softmax(q @ self.k(kv).transpose(-2, -1) * self.scale, -1)
        return w @ self.v(kv), w

class CoralHead(nn.Module):
    def __init__(self, d, n=5):
        super().__init__(); self.fc = nn.Linear(d, 1, bias=False); self.bias = nn.Parameter(torch.zeros(n - 1))
    def forward(self, h): return self.fc(h) + self.bias

class FlexMultiTask(nn.Module):
    """
    tasks       : "both" | "dr" | "dme"
    proj        : True  -> task-specific linear projections;  False -> identity
    interaction : "bi"       bidirectional cross-task attention (proposed)
                  "none"     no attention
                  "dr<-dme"  DR queries DME tokens only        "dme<-dr"  DME queries DR tokens only
                  "self"     self-attention inside each branch (capacity-matched control)
    dr_head     : "coral" (ordinal, proposed) | "softmax" (categorical)
    Shared-backbone dual-head baseline = proj=False, interaction="none".
    """
    def __init__(self, cfg=CFG, ckpt=None, tasks="both", proj=True, interaction="bi",
                 dr_head="coral", dropout=None):
        super().__init__()
        D = cfg.feat_dim
        self.tasks, self.dr_head_type, self.interaction = tasks, dr_head, interaction
        self.has_dr, self.has_dme = tasks in ("both", "dr"), tasks in ("both", "dme")
        self.backbone = load_imagenet_weights(EfficientVMambaS(drop_path=cfg.drop_path), ckpt)
        self.proj_dr  = (nn.Linear(D, D) if proj else nn.Identity()) if self.has_dr  else None
        self.proj_dme = (nn.Linear(D, D) if proj else nn.Identity()) if self.has_dme else None
        use_dr_att  = self.has_dr  and interaction in ("bi", "dr<-dme", "self")
        use_dme_att = self.has_dme and interaction in ("bi", "dme<-dr", "self")
        self.att_dr  = TokenAttention(D, cfg.ln_eps) if use_dr_att  else None
        self.att_dme = TokenAttention(D, cfg.ln_eps) if use_dme_att else None
        self.drop = nn.Dropout(cfg.dropout if dropout is None else dropout)
        if self.has_dr:  self.head_dr  = CoralHead(D, cfg.n_dr) if dr_head == "coral" else nn.Linear(D, cfg.n_dr)
        if self.has_dme: self.head_dme = nn.Linear(D, cfg.n_dme)
        for n, m in self.named_modules():                                          # Xavier for new layers
            if not n.startswith("backbone") and isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x, return_all=False):
        Fs = self.backbone(x).flatten(2).transpose(1, 2)                           # B,49,D
        F_dr  = self.proj_dr(Fs)  if self.has_dr  else None
        F_dme = self.proj_dme(Fs) if self.has_dme else None
        kv_dme = F_dr if self.interaction in ("bi", "dme<-dr") else F_dme          # what each branch attends to
        kv_dr  = F_dme if self.interaction in ("bi", "dr<-dme") else F_dr
        out = {"dr_logits": None, "dme_logits": None}
        if self.has_dr:
            h = F_dr + self.att_dr(F_dr, kv_dr)[0] if self.att_dr is not None else F_dr
            out["dr_logits"] = self.head_dr(self.drop(h.mean(1)))
        if self.has_dme:
            h = F_dme + self.att_dme(F_dme, kv_dme)[0] if self.att_dme is not None else F_dme
            out["dme_logits"] = self.head_dme(self.drop(h.mean(1)))
        return out



In [ ]:


def init_env():
    """Initialize pretrained backbone and random seed."""
    resolve_backbone_ckpt()
    set_seed(CFG.seed)

    print(
        "Device:",
        device,
        torch.cuda.get_device_name(0) if device.type == "cuda" else ""
    )


def cached(path, fn):
    """Load previously saved results or execute and cache new results."""

    if os.path.exists(path):
        print(f"Loading cached result: {os.path.basename(path)}")
        with open(path, "r") as f:
            return json.load(f)

    print(f"Running experiment: {os.path.basename(path)}")

    result = fn()

    with open(path, "w") as f:
        json.dump(result, f, indent=1)

    return result


print("Helper functions initialized successfully.")


In [ ]:
# HYPERPARAMETER SEARCH: Random search, Optuna-TPE, HEO and AHEO

LOWER = np.array([-5.0,  8.0, -6.0, 0.1,  50.0])
UPPER = np.array([-3.0, 32.0, -3.0, 0.5, 150.0])
M_POP, PROXY_FRAC = 10, 0.2                         # proxy training = round(0.2 * E) epochs
LEVY_BETA, LEVY_SCALE, INIT_VEL = 1.5, 0.01, 0.10   # HEO settings
K_D, K_I, G_MIN, G_MAX, EPS = 0.50, 1.00, 0.50, 1.50, 1e-8   # AHEO settings
DEFAULT_HP = dict(lr=5e-4, batch_size=16, weight_decay=1e-5, dropout=0.30, epochs=100)

def decode(x):
    return {"lr": float(10 ** x[0]), "batch_size": int(round(x[1])), "weight_decay": float(10 ** x[2]),
            "dropout": float(x[3]), "epochs": int(round(x[4]))}

def repair(x):
    x = np.clip(np.asarray(x, float), LOWER, UPPER)
    x[1], x[4] = round(x[1]), round(x[4])
    return x

def sample_init(seed, m=M_POP):
    """The same ten initial candidates are shared by all optimizers for a given seed."""
    rng = np.random.default_rng(seed)
    return [repair(rng.uniform(LOWER, UPPER)) for _ in range(m)]

def make_fitness(D):
    def fitness(x, seed):
        hp = decode(x); hp["epochs"] = max(1, int(round(PROXY_FRAC * hp["epochs"])))
        r = train_eval(proposed_fn, D, hp, seed=seed, eval_test=False, verbose=False)
        return float(r["best_J"]), hp["epochs"]
    return fitness

class Recorder:

    def __init__(self, name, seed, budget, path, fitness):
        self.name, self.seed, self.budget, self.path, self.fitness = name, seed, budget, path, fitness
        self.log = json.load(open(path)) if os.path.exists(path) else []
        self.calls, self.best_f, self.best_x, self.curve, self.gammas = 0, float("inf"), None, [], []
    def __call__(self, x):
        if self.calls >= self.budget: raise StopIteration
        x = [float(v) for v in x]
        if self.calls < len(self.log) and np.allclose(self.log[self.calls]["x"], x, atol=1e-6):
            f = self.log[self.calls]["J"]
        else:
            self.log = self.log[:self.calls]; t = time.time()
            f, proxy = self.fitness(np.array(x), self.seed)
            self.log.append({"eval": self.calls + 1, "x": x, "hp": decode(x), "proxy_epochs": proxy, "J": f,
                             "minutes": (time.time() - t) / 60})
            json.dump(self.log, open(self.path, "w"), indent=1)
            print(f"   [{self.name} s{self.seed}] eval {self.calls+1:3d}/{self.budget} | J {f:.4f} | "
                  f"{self.log[-1]['minutes']:.1f} min", flush=True)
        self.calls += 1
        if f < self.best_f: self.best_f, self.best_x = f, np.array(x)
        self.curve.append(self.best_f)
        return f

def levy_step(rng, size, beta=LEVY_BETA):
    sigma = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
             (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)
    return rng.normal(0, sigma, size) / np.abs(rng.normal(0, 1, size)) ** (1 / beta)

def run_random(rec, rng, init_pop):
    for x in init_pop: rec(x)
    while rec.calls < rec.budget: rec(repair(rng.uniform(LOWER, UPPER)))

def run_optuna(rec, seed, init_pop):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=len(init_pop)))
    names = ["lr", "bs", "wd", "dr", "ep"]
    for x in init_pop: study.enqueue_trial({n: (int(v) if n in ("bs", "ep") else float(v)) for n, v in zip(names, x)})
    def objective(trial):
        x = [trial.suggest_float("lr", -5, -3), trial.suggest_int("bs", 8, 32), trial.suggest_float("wd", -6, -3),
             trial.suggest_float("dr", 0.1, 0.5), trial.suggest_int("ep", 50, 150)]
        return rec(repair(x))
    study.optimize(objective, n_trials=rec.budget)

def run_heo(rec, rng, init_pop, adaptive, T):

    span = UPPER - LOWER
    pop = np.array([repair(x) for x in init_pop]); M = len(pop)
    vel = rng.uniform(-INIT_VEL, INIT_VEL, pop.shape) * span
    fit = np.array([rec(x) for x in pop])
    f_prev, I_t = None, 0.0
    for t in range(1, T + 1):
        f_best, x_best = float(fit.min()), pop[int(fit.argmin())].copy()
        if f_prev is not None: I_t = abs(f_prev - f_best) / (abs(f_prev) + EPS)
        f_prev = f_best
        gamma, D_t = 1.0, float("nan")
        if adaptive:
            Z = (pop - LOWER) / (span + EPS)
            D_t = float(np.clip(np.linalg.norm(Z - Z.mean(0), axis=1).mean() / (math.sqrt(5) / 2), 0, 1))
            gamma = float(np.clip(1 + K_D * (1 - D_t) - K_I * I_t, G_MIN, G_MAX))
        decay = 1 - (t - 1) / T
        vel = decay * vel + rng.random(pop.shape) * (x_best - pop)
        new = pop + gamma * vel + LEVY_SCALE * span * levy_step(rng, pop.shape)
        new = np.array([repair(x) for x in new])
        new_fit = np.array([rec(x) for x in new])
        allp, allv, allf = np.vstack([pop, new]), np.vstack([vel, vel]), np.concatenate([fit, new_fit])
        keep = np.argsort(allf)[:M]
        pop, vel, fit = allp[keep], allv[keep], allf[keep]
        rec.gammas.append({"iter": t, "gamma": gamma, "D": D_t, "I": I_t})

def run_search(name, seed, budget, D, init_pop, tag=""):

    path = os.path.join(OUT_DIR, f"search_{name}_seed{seed}_b{budget}{tag}.json")
    rec = Recorder(name, seed, budget, path, make_fitness(D))
    rng = np.random.default_rng(seed + 1000)
    T = (budget - M_POP) // M_POP
    if   name == "Random":     run_random(rec, rng, init_pop)
    elif name == "Optuna-TPE": run_optuna(rec, seed, init_pop)
    elif name == "HEO":        run_heo(rec, rng, init_pop, adaptive=False, T=T)
    elif name == "AHEO":       run_heo(rec, rng, init_pop, adaptive=True,  T=T)
    return {"name": name, "seed": seed, "best_J": rec.best_f, "best_x": rec.best_x.tolist(),
            "best_hp": decode(rec.best_x), "curve": rec.curve,
            "minutes": float(sum(e["minutes"] for e in rec.log)), "gammas": rec.gammas}



In [ ]:

SEED       = 42
BUDGET     = M_POP + 10 * M_POP
OPTIMIZERS = ["Random", "Optuna-TPE", "HEO", "AHEO"]

def sci(v, digits=1):
    m, e = f"{v:.{digits}e}".split("e"); m = m.rstrip("0").rstrip(".")
    return f"{m} x 10^{int(e)}"

def main():
    init_env(); D = build_data()
    init_pop = sample_init(SEED)
    # searches
    search = {n: cached(os.path.join(OUT_DIR, f"main_{n}_seed{SEED}_RESULT.json"),
                        lambda n=n: run_search(n, SEED, BUDGET, D, init_pop)) for n in OPTIMIZERS}
    # full-length retraining of the default and every selected configuration, single test evaluation
    cfgs = {"Default": DEFAULT_HP, **{n: search[n]["best_hp"] for n in OPTIMIZERS}}
    full = {n: cached(os.path.join(OUT_DIR, f"fulltrain_{n}.json"),
                      lambda hp=hp: train_eval(proposed_fn, D, hp, seed=SEED)) for n, hp in cfgs.items()}

    strat = {"Without optimization": "Default", "Original HEO": "HEO", "Proposed AHEO": "AHEO"}
    t7 = pd.DataFrame([table_row(k, full[v]) for k, v in strat.items()]).rename(columns={"Configuration": "Optimization Strategy"})
    t7.to_csv(os.path.join(OUT_DIR, "table7_optimizer_ablation.csv"), index=False)

    cols = ["Default"] + OPTIMIZERS
    rows = [["Learning rate", "1e-5 - 1e-3", "log10-uniform"] + [sci(cfgs[c]["lr"]) for c in cols],
            ["Batch size", "8-32", "integer"] + [str(cfgs[c]["batch_size"]) for c in cols],
            ["Weight decay", "1e-6 - 1e-3", "log10-uniform"] + [sci(cfgs[c]["weight_decay"]) for c in cols],
            ["Dropout", "0.1-0.5", "continuous"] + [f"{cfgs[c]['dropout']:.2f}" for c in cols],
            ["Epochs", "50-150", "integer"] + [str(cfgs[c]["epochs"]) for c in cols],
            ["Val. J after full-length retraining", "-", "-"] + [f"{full[c]['best_J']:.4f}" for c in cols]]
    t8 = pd.DataFrame(rows, columns=["Hyperparameter", "Search range", "Encoding", "Default", "Random search", "Optuna-TPE", "HEO", "AHEO"][:3 + len(cols)])
    t8.to_csv(os.path.join(OUT_DIR, "table8_search_space.csv"), index=False)

    t9 = pd.DataFrame([{"Optimizer": n, "Population": "none" if n in ("Random", "Optuna-TPE") else M_POP,
                        "Fitness evaluations": BUDGET, "GPU time of search (min)": round(search[n]["minutes"], 1)}
                       for n in OPTIMIZERS])
    t9.to_csv(os.path.join(OUT_DIR, "table9_search_cost.csv"), index=False)
    # comparison of AHEO vs HEO
    A, H = full["AHEO"], full["HEO"]
    print("\nTable \n", t7.to_string(index=False)); print("\nTable \n", t8.to_string(index=False)); print("\nTable \n", t9.to_string(index=False))
    print(f"\nMCC gain AHEO vs HEO: DR {100*(A['test']['dr']['mcc']-H['test']['dr']['mcc']):+.2f} pp | "
          f"DME {100*(A['test']['dme']['mcc']-H['test']['dme']['mcc']):+.2f} pp")


main()



Table 

Optimization Strategy  Val. DR F1 (%)  Val. DME F1 (%)  Val. J  Test DR Acc (%)  Test DR Kappa (%)  Test DR MCC (%)  Test DME Acc (%)  Test DME Kappa (%)  Test DME MCC (%)
 Without optimization           93.98            95.90  0.0506            94.17              92.17            92.20             96.12               93.33             93.33
         Original HEO           95.85            96.82  0.0367            96.12              94.75            94.77             97.09               95.03             95.04
        Proposed AHEO           97.72            98.64  0.0182            98.06              97.38            97.38             99.03               98.32             98.34

Table 

                     Hyperparameter Search range      Encoding   Default Random search  Optuna-TPE         HEO      AHEO
                      Learning rate  1e-5 - 1e-3 log10-uniform 5 x 10^-4   3.1 x 10^-4 1.6 x 10^-4 2.2 x 10^-4 1 x 10^-4
                         Batch size         8-32    